# zanembed-v01: Embedding Model for Kazakh Law

This notebook provides the data pipeline for **zanembed-v01** — a domain-specific text embedding model
fine-tuned for Kazakhstani legal documents. It is built on top of **zembed1** and adapted to the
structure and vocabulary of Kazakhstan's legislation.

Alisher Sailaubek  
alisher@caricnomix.tech

## What is zanembed-v01?

**zanembed-v01** is a bilingual (Russian/Kazakh) embedding model that produces high-quality semantic
representations of Kazakhstani legal texts. It addresses the poor retrieval performance of general-purpose
embeddings on domain-specific legal corpora sourced from Kazakhstan's official legal database
[adilet.zan.kz](https://adilet.zan.kz).

## Base model: zembed1

zembed1 is a multilingual sentence embedding model pre-trained on Central Asian and CIS-region corpora.
zanembed-v01 extends zembed1 via continued pre-training and fine-tuning on legal document pairs from
adilet.zan.kz, covering laws, presidential decrees, government resolutions, and normative acts.

## Datasets

### Full corpus — `law_dataset_rus.json`
All scraped documents across all spheres (unbounded).

### Training dataset — `laws_dataset_training.json`
Balanced 1 000-document subset used for fine-tuning. ~250 documents per sphere.

Documents are sourced from the Russian-language section of adilet.zan.kz using sphere search filters:

| Sphere | Search filter |
|---|---|
| Farming | `ir=1_005` |
| Labor | `ir=1_002` |
| Finance | `ir=1_006` |
| Sales and others | `ir=1_019` |

Each entry:
```json
{
  "url": "https://adilet.zan.kz/rus/docs/...",
  "title": "Document title",
  "doc_type": "Закон | Постановление | Указ | ...",
  "date": "YYYY-MM-DD",
  "number": "document number",
  "authority": "issuing body",
  "status": "Active | Repealed",
  "sphere": "Farming | Labor | Finance | Sales and others",
  "content": "full text of the legal act"
}
```

## Pipeline (this notebook)

1. **Scraping** — legal acts from [adilet.zan.kz](https://adilet.zan.kz) (RU) → `law_dataset_rus.json`
2. **Training subset (RU)** — balanced ~1 000 docs → `laws_dataset_training.json`
3. **Kazakh corpus** — parallel `/kaz/` scrape → `laws_dataset_kaz.json`
4. **Triplets** — semantic pairs + hard negatives → `triplets_raw.json` (cleaned in place)
5. **LLM labeling** — Kimi via Groq → `triplets_labeled.json`
6. **Fine-tuning** — triplet loss + LoRA on `zeroentropy/zembed-1` → `zanembed-v01-lora/`


In [ ]:
!pip install python-dotenv qdrant-client huggingface_hub openai groq fastembed tiktoken requests beautifulsoup4 anthropic


# Web Scraper

We scrape Kazakhstan's official legal database [adilet.zan.kz/rus](https://adilet.zan.kz/rus)
using **requests + BeautifulSoup** — no browser required.

Documents are collected per legal sphere using adilet's built-in search filters (`ir` parameter).
Each document page is fetched alongside its `/info` tab, which provides structured metadata.

### Metadata extracted per document

| Field | Source |
|---|---|
| `title` | `<h1>` on the document page |
| `doc_type` | "Форма акта" row in `/info` table |
| `date` | "Дата принятия" row in `/info` table |
| `number` | "Регистрационный номер" row in `/info` table |
| `authority` | "Орган" row in `/info` table |
| `status` | "Статус" row in `/info` table; fallback: "Утратил силу" banner on page |
| `sphere` | Derived from the search filter used to discover the document |
| `content` | `<article>` body text |

Results are saved to `law_dataset_rus.json`. A checkpoint file is written every 50 documents
so interrupted runs resume without re-scraping.

In [ ]:
# Web scraper for adilet.zan.kz/rus
# Uses requests + BeautifulSoup — no browser dependencies required.
#
# Output schema (dataset_rus.json):
#   url, title, doc_type, date, number, authority, status, sphere, content
#
# status  → "Active" | "Repealed"
# sphere  → legal domain derived from the search category used to find the document

import json, os, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin
import urllib3
import requests
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Sphere definitions ────────────────────────────────────────────────────
# Each sphere maps to its adilet.zan.kz search filter URL.
SPHERES = {
    "Farming":        "https://adilet.zan.kz/rus/search/docs/ir=1_005",
    "Labor":          "https://adilet.zan.kz/rus/search/docs/ir=1_002",
    "Finance":        "https://adilet.zan.kz/rus/search/docs/ir=1_006",
    "Sales and others": "https://adilet.zan.kz/rus/search/docs/ir=1_019",
}

OUTPUT_FILE = "law_dataset_rus.json"
CHECKPOINT  = "law_dataset_rus_checkpoint.json"
MAX_CONCURRENCY = 4
DELAY       = 0.35
MAX_RETRIES = 3
TIMEOUT     = 30
MAX_PAGE_NUM = 25000

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


def make_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    s.verify = False  # adilet.zan.kz uses a non-standard CA
    return s


def fetch(session, url, retries=MAX_RETRIES):
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            r.encoding = r.apparent_encoding or "utf-8"
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            wait = 2 ** attempt
            print(f"  [retry {attempt+1}/{retries}] {url} — {e} (wait {wait}s)")
            time.sleep(wait)
    print(f"  [FAIL] {url}")
    return None


def clean(text):
    return re.sub(r"\s+", " ", text or "").strip()


def get_doc_links(session, start_url, max_pages=None):
    """
    Paginate from start_url and return all document URLs found.
    start_url is the sphere search URL, e.g.:
        https://adilet.zan.kz/rus/search/docs/ir=1_005
    Pagination follows a.nextpostslink on each page.
    """
    base = "https://adilet.zan.kz"
    # Append pagesize to the start URL
    current = start_url + "&pagesize=100" if "&" in start_url or "?" in start_url else start_url + "&pagesize=100"
    # Handle the case where start_url has no query-like params yet
    if "pagesize" not in current:
        current = start_url + "&pagesize=100"

    links = []
    pages_done = 0

    while current:
        print(f"  [listing] {current}")
        soup = fetch(session, current)
        if soup is None:
            print("  [!] Failed to load listing page, stopping.")
            break

        for a in soup.select("h4.post_header a[href]"):
            href = a["href"]
            full = urljoin(base, href) if not href.startswith("http") else href
            links.append(full)

        page_count = len(soup.select("h4.post_header a"))
        print(f"    -> {page_count} docs (running total: {len(links)})")

        pages_done += 1
        if max_pages and pages_done >= max_pages:
            break

        # Follow the true "next" link (smallest page number below cap)
        next_url = None
        for a in soup.select("a.nextpostslink[href]"):
            href = a["href"]
            m = re.search(r"page=(\d+)", href)
            if not m or int(m.group(1)) > MAX_PAGE_NUM:
                continue
            candidate = urljoin(base, href) if not href.startswith("http") else href
            if next_url is None:
                next_url = candidate
            else:
                cur_n = int(re.search(r"page=(\d+)", next_url).group(1))
                if int(m.group(1)) < cur_n:
                    next_url = candidate

        current = next_url
        if current:
            time.sleep(DELAY)

    seen, out = set(), []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def parse_status(soup_main, info_rows):
    REPEALED = ["утратил силу", "признан утратившим", "недействующий"]
    for key, val in info_rows:
        if "статус" in key.lower():
            return "Repealed" if any(kw in val.lower() for kw in REPEALED) else "Active"
    if soup_main:
        if any(kw in soup_main.get_text().lower() for kw in REPEALED):
            return "Repealed"
    return "Active"


def scrape_document(url, sphere=""):
    session = make_session()

    soup = fetch(session, url)
    if soup is None:
        return None

    h1 = soup.find("h1")
    title = clean(h1.get_text()) if h1 else ""

    content = ""
    for sel in ["article", "div.document-text", "div#document-text", "div.post-content"]:
        el = soup.select_one(sel)
        if el:
            content = clean(el.get_text())
            break
    if len(content) < 100:
        return None

    doc_type = date = number = authority = ""
    info_rows = []
    info_soup = fetch(session, url.rstrip("/") + "/info")
    if info_soup:
        table = info_soup.find("table", id="ethernatable") or info_soup.find("table")
        if table:
            for row in table.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) < 2:
                    continue
                key = clean(cells[0].get_text())
                val = clean(cells[1].get_text())
                info_rows.append((key, val))
                k = key.lower()
                if ("дата принятия" in k or "дата" in k) and not date:
                    date = val
                elif ("форма акта" in k or "форма" in k) and not doc_type:
                    doc_type = val
                elif ("регистрационный номер" in k or "номер" in k) and not number:
                    number = val
                elif "орган" in k and not authority:
                    authority = val

    status = parse_status(soup, info_rows)
    tag = "R" if status == "Repealed" else "A"
    print(f"[{tag}][{sphere}] {doc_type or '?'}: {title[:50]} ({date})")

    return {
        "url": url,
        "title": title,
        "doc_type": doc_type,
        "date": date,
        "number": number,
        "authority": authority,
        "status": status,
        "sphere": sphere,
        "content": content,
    }


def scrape_zan(spheres=None, max_pages=None, workers=MAX_CONCURRENCY,
               output_file=OUTPUT_FILE, checkpoint_file=CHECKPOINT):
    """
    Scrape all spheres defined in SPHERES (or pass a custom dict).
    Each document is tagged with its sphere name.
    """
    if spheres is None:
        spheres = SPHERES

    # Load checkpoint
    done = {}
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, encoding="utf-8") as f:
            for doc in json.load(f):
                done[doc["url"]] = doc
        print(f"[resume] {len(done)} docs from checkpoint.")

    session = make_session()

    # Collect links per sphere
    sphere_links = []  # list of (url, sphere)
    for sphere_name, search_url in spheres.items():
        print(f"\n[*] Sphere: {sphere_name}")
        links = get_doc_links(session, search_url, max_pages=max_pages)
        print(f"  -> {len(links)} unique links")
        for u in links:
            sphere_links.append((u, sphere_name))

    # Deduplicate by URL (a doc can appear in multiple spheres — keep first occurrence)
    seen_urls = set(done.keys())
    todo = []
    for url, sphere in sphere_links:
        if url not in seen_urls:
            seen_urls.add(url)
            todo.append((url, sphere))

    print(f"\n[*] Total to scrape: {len(todo)} (already done: {len(done)})\n")

    results = dict(done)
    completed = 0

    def worker(args):
        url, sphere = args
        time.sleep(DELAY)
        return url, scrape_document(url, sphere)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(worker, item): item for item in todo}
        for future in as_completed(futures):
            url, doc = future.result()
            if doc:
                results[url] = doc
            completed += 1
            if completed % 50 == 0:
                snap = list(results.values())
                with open(checkpoint_file, "w", encoding="utf-8") as f:
                    json.dump(snap, f, ensure_ascii=False, indent=2)
                print(f"  [checkpoint] {len(snap)} docs | {completed}/{len(todo)} done")

    dataset = list(results.values())
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"\n✅ {len(dataset)} documents saved to {output_file}")
    return dataset


In [ ]:
# Test — 2 listing pages per sphere (~200 docs total)
dataset = scrape_zan(max_pages=2)

from collections import Counter
print('Docs per sphere:', Counter(d['sphere'] for d in dataset))
print('Active/Repealed:', Counter(d['status'] for d in dataset))

for doc in dataset[:3]:
    print(f"\n=== [{doc['status']}][{doc['sphere']}] {doc['doc_type']}: {doc['title'][:60]} ===")
    print(f"Date: {doc['date']} | Number: {doc['number']} | Authority: {doc['authority'][:50]}")
    print(doc['content'][:300], "...")

In [ ]:
print(dataset)

In [ ]:
# Full scrape — all spheres, all pages
dataset = scrape_zan(max_pages=None)

from collections import Counter
print('Docs per sphere:', Counter(d['sphere'] for d in dataset))
print('Active/Repealed:', Counter(d['status'] for d in dataset))

# Training Dataset — `laws_dataset_training.json`

Builds a balanced 1 000-document training set from the scraped corpus.

- **Farming** — sampled from `law_dataset_rus_checkpoint.json` (already scraped)
- **Labor / Finance / Sales and others** — quick scrape: 3 listing pages × ~100 docs/page per sphere

Target: **250 documents per sphere**, 1 000 total.

Output: `laws_dataset_training.json`


In [ ]:
import json, os, re, time, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

CHECKPOINT_FILE   = "law_dataset_rus_checkpoint.json"
TRAINING_OUTPUT   = "laws_dataset_training.json"
TARGET_PER_SPHERE = 250
MAX_LIST_PAGES    = 3

SPHERES_TO_SCRAPE = {
    "Labor":            "https://adilet.zan.kz/rus/search/docs/ir=1_002",
    "Finance":          "https://adilet.zan.kz/rus/search/docs/ir=1_006",
    "Sales and others": "https://adilet.zan.kz/rus/search/docs/ir=1_019",
}

# ── Step 1: sample Farming from checkpoint ──────────────────────────────
print("Step 1 — loading Farming records from checkpoint")
farming_records = []
if os.path.exists(CHECKPOINT_FILE):
    decoder = json.JSONDecoder()
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        raw = f.read()
    pos = raw.index("[") + 1
    while pos < len(raw):
        while pos < len(raw) and raw[pos] in " \t\n\r,":
            pos += 1
        if pos >= len(raw) or raw[pos] == "]":
            break
        try:
            obj, end = decoder.raw_decode(raw, pos)
            if obj.get("sphere") == "Farming":
                farming_records.append(obj)
            pos = end
        except json.JSONDecodeError:
            break

random.seed(42)
farming_sample = random.sample(farming_records, min(TARGET_PER_SPHERE, len(farming_records)))
print(f"  {len(farming_records)} Farming records found, sampled {len(farming_sample)}")

# ── Step 2: scrape remaining spheres ────────────────────────────────────
all_records = list(farming_sample)

for sphere_name, search_url in SPHERES_TO_SCRAPE.items():
    print(f"\nScraping {sphere_name}...")
    session = make_session()
    links = get_doc_links(session, search_url, max_pages=MAX_LIST_PAGES)
    random.shuffle(links)
    batch = links[: TARGET_PER_SPHERE * 2]

    sphere_docs = []

    def _worker(url):
        time.sleep(DELAY)
        return scrape_document(url, sphere_name)

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as pool:
        futures = [pool.submit(_worker, u) for u in batch]
        for fut in as_completed(futures):
            doc = fut.result()
            if doc:
                sphere_docs.append(doc)
            if len(sphere_docs) >= TARGET_PER_SPHERE:
                break

    all_records.extend(sphere_docs[:TARGET_PER_SPHERE])
    print(f"  {len(sphere_docs[:TARGET_PER_SPHERE])} docs collected for {sphere_name}")

# ── Step 3: save ────────────────────────────────────────────────────────
with open(TRAINING_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

dist = Counter(d["sphere"] for d in all_records)
print(f"\nSaved {len(all_records)} records to {TRAINING_OUTPUT}")
print("Distribution:", dict(dist))


# Kazakh Dataset — `laws_dataset_kaz.json`

Scrapes the **Kazakh-language** (`/kaz/`) section of adilet.zan.kz using the same four legal spheres.
This dataset mirrors the Russian training set and is intended for bilingual fine-tuning of zanembed-v01.

## Why a separate Kazakh corpus?

Russian and Kazakh versions of the same law are independent translations — they share structure
and legal semantics but differ significantly in vocabulary and morphology. Training on both languages
helps the embedding model align legal concepts across languages, which is critical for cross-lingual
retrieval in Kazakhstan's bilingual legal system.

## Sphere mapping

| Sphere | Search filter | URL |
|---|---|---|
| Farming | `ir=1_005` | `/kaz/search/docs/ir=1_005` |
| Labor | `ir=1_002` | `/kaz/search/docs/ir=1_002` |
| Finance | `ir=1_006` | `/kaz/search/docs/ir=1_006` |
| Sales and others | `ir=1_019` | `/kaz/search/docs/ir=1_019` |

## Schema

Same as the Russian dataset with one addition:

```json
{
  "url": "https://adilet.zan.kz/kaz/docs/...",
  "title": "Документ атауы",
  "doc_type": "Заң | Қаулы | Жарлық | ...",
  "date": "DD.MM.YYYY",
  "number": "document number",
  "authority": "issuing body",
  "status": "Active | Repealed",
  "sphere": "Farming | Labor | Finance | Sales and others",
  "language": "kaz",
  "content": "full text in Kazakh"
}
```

## Output

- `laws_dataset_kaz.json` — standalone Kazakh corpus (~1 000 docs, 250 per sphere)
- Can be merged with `laws_dataset_training.json` for a combined bilingual training set


In [ ]:
import json, time, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

KAZ_OUTPUT        = "laws_dataset_kaz.json"
KAZ_TARGET        = 250   # docs per sphere
KAZ_MAX_PAGES     = 3

KAZ_SPHERES = {
    "Farming":          "https://adilet.zan.kz/kaz/search/docs/ir=1_005",
    "Labor":            "https://adilet.zan.kz/kaz/search/docs/ir=1_002",
    "Finance":          "https://adilet.zan.kz/kaz/search/docs/ir=1_006",
    "Sales and others": "https://adilet.zan.kz/kaz/search/docs/ir=1_019",
}

kaz_records = []

for sphere_name, search_url in KAZ_SPHERES.items():
    print(f"\n[kaz] Scraping {sphere_name}...")
    session = make_session()
    links = get_doc_links(session, search_url, max_pages=KAZ_MAX_PAGES)
    random.shuffle(links)
    batch = links[: KAZ_TARGET * 2]
    sphere_docs = []

    def _kaz_worker(url):
        time.sleep(DELAY)
        doc = scrape_document(url, sphere_name)
        if doc:
            doc["language"] = "kaz"
        return doc

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as pool:
        futures = [pool.submit(_kaz_worker, u) for u in batch]
        for fut in as_completed(futures):
            doc = fut.result()
            if doc:
                sphere_docs.append(doc)
            if len(sphere_docs) >= KAZ_TARGET:
                break

    kaz_records.extend(sphere_docs[:KAZ_TARGET])
    print(f"  collected {len(sphere_docs[:KAZ_TARGET])} docs")

with open(KAZ_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(kaz_records, f, ensure_ascii=False, indent=2)

dist = Counter(d["sphere"] for d in kaz_records)
print(f"\nSaved {len(kaz_records)} Kazakh records to {KAZ_OUTPUT}")
print("Distribution:", dict(dist))

# ── Optional: merge with Russian training set ────────────────────────────
# with open(TRAINING_OUTPUT) as f:
#     rus_records = json.load(f)
# combined = rus_records + kaz_records
# with open("laws_dataset_bilingual.json", "w", encoding="utf-8") as f:
#     json.dump(combined, f, ensure_ascii=False, indent=2)
# print(f"Combined bilingual dataset: {len(combined)} docs")


# LoRA Fine-tuning Pipeline — ZanEmbed

Fine-tunes **Zembed-1** on Kazakhstani legal documents using LoRA adapters and triplet loss.
The pipeline has 5 stages: **Semantic Pair Sampling → Content Cleaning → Kimi K2.5 Labeling → Triplet Dataset → LoRA Training**.

---

## Inputs

| File | Docs | Language |
|---|---|---|
| `laws_dataset_training.json` | 1 000 | Russian (`rus`) |
| `laws_dataset_kaz.json` | 1 000 | Kazakh (`kaz`) |

527 documents share the same doc ID across both files → ready-made **cross-lingual positive pairs**.

---

## Stage 2 — Semantic Pair Sampler

Generates ~3 000 raw triplets of the form **(anchor, positive, hard_negative)**.

| Pair type | How | Example |
|---|---|---|
| Cross-lingual positive | Same doc ID in `/rus/` and `/kaz/` | Finance law RU ↔ Finance law KAZ |
| Same-sphere positive (RU) | Different doc, same sphere | Labor law A ↔ Labor law B |
| Same-sphere positive (KAZ) | Different doc, same sphere, Kazakh only | Farming law A ↔ Farming law B |
| Hard negative | Different sphere | Finance law ↔ Farming law |

Output: `triplets_raw.json`

---

## Stage 2.5 — Content Cleaner

Filters `triplets_raw.json` before LLM labeling to ensure only semantically rich triplets proceed.

Drops triplets where any leg:
- Is under 300 characters or 60 words (stubs, headers)
- Has >40% of lines matching legal reference patterns (`№`, `статья N`, `35-бап`, Kazakh year refs) — mostly amendment laws with no prose content

Also deduplicates by anchor+positive URL pair.

Output: overwrites `triplets_raw.json` with clean subset.

---

## Stage 3 — Kimi K2.5 Reasoner (via Groq)

Each cleaned triplet is passed to **Kimi K2.5** with a structured prompt showing all three documents.
The model labels the anchor→positive relationship and scores confidence.

Filtered out if:
- `confidence < 0.65`
- `relationship == "unrelated"`
- `hard_negative_distinct == false`
- `deprecated == true`

Runs async with concurrency=8 and checkpoints every 200 items to `triplets_labeled.json`.

Output: `triplets_labeled.json`

---

## Stage 4 — Triplet Loss Training

Anchors are pulled toward positives and pushed away from hard negatives in embedding space.
Uses `sentence-transformers` `TripletLoss` with cosine distance.

## Stage 5 — LoRA Config

Applied to all linear layers of Zembed-1:

```
r = 32,  alpha = 64
target_modules: q_proj, v_proj, k_proj, o_proj, gate_proj, up_proj, down_proj
~0.5% trainable params (~20M)
```

Output: `zanembed-v01-lora/` — Zembed-1 + LoRA adapter (Legal-RK)

In [ ]:
# Stage 2: Semantic Pair Sampler
# Produces (anchor, positive, hard_negative) triplets from both datasets.

import json, re, random
from collections import defaultdict, Counter

with open("laws_dataset_training.json", encoding="utf-8") as f:
    rus_docs = json.load(f)
with open("laws_dataset_kaz.json", encoding="utf-8") as f:
    kaz_docs = json.load(f)

def doc_id(url):
    m = re.search(r'/docs/([A-Z]\d+)', url)
    return m.group(1) if m else None

rus_by_id     = {doc_id(d['url']): d for d in rus_docs if doc_id(d['url'])}
kaz_by_id     = {doc_id(d['url']): d for d in kaz_docs if doc_id(d['url'])}
rus_by_sphere = defaultdict(list)
kaz_by_sphere = defaultdict(list)
for d in rus_docs: rus_by_sphere[d['sphere']].append(d)
for d in kaz_docs: kaz_by_sphere[d['sphere']].append(d)

cross_ids = list(set(rus_by_id) & set(kaz_by_id))
print(f"Cross-lingual pairs available: {len(cross_ids)}")

SPHERES = ["Farming", "Labor", "Finance", "Sales and others"]
random.seed(42)
triplets = []

# ── Type 1: RU anchor → KAZ positive ─────────────────────────────────────
for did in random.sample(cross_ids, min(1000, len(cross_ids))):
    anchor   = rus_by_id[did]
    positive = kaz_by_id[did]
    sphere   = anchor['sphere']
    neg_sphere = random.choice([s for s in SPHERES if s != sphere])
    hard_neg = random.choice(rus_by_sphere[neg_sphere])
    triplets.append({
        "anchor":          anchor['content'][:2000],
        "positive":        positive['content'][:2000],
        "hard_negative":   hard_neg['content'][:2000],
        "anchor_sphere":   sphere,
        "negative_sphere": neg_sphere,
        "pair_type":       "cross_lingual_ru_kaz",
        "anchor_url":      anchor['url'],
        "positive_url":    positive['url'],
    })

# ── Type 2: KAZ anchor → RU positive (reverse) ───────────────────────────
for did in random.sample(cross_ids, min(1000, len(cross_ids))):
    anchor   = kaz_by_id[did]
    positive = rus_by_id[did]
    sphere   = anchor['sphere']
    neg_sphere = random.choice([s for s in SPHERES if s != sphere])
    hard_neg = random.choice(kaz_by_sphere[neg_sphere])
    triplets.append({
        "anchor":          anchor['content'][:2000],
        "positive":        positive['content'][:2000],
        "hard_negative":   hard_neg['content'][:2000],
        "anchor_sphere":   sphere,
        "negative_sphere": neg_sphere,
        "pair_type":       "cross_lingual_kaz_ru",
        "anchor_url":      anchor['url'],
        "positive_url":    positive['url'],
    })

# ── Type 3: same-sphere RU ────────────────────────────────────────────────
for sphere in SPHERES:
    pool = rus_by_sphere[sphere]
    pairs = [(pool[i], pool[j]) for i in range(len(pool)) for j in range(i+1, min(i+4, len(pool)))]
    for anchor, positive in random.sample(pairs, min(250, len(pairs))):
        neg_sphere = random.choice([s for s in SPHERES if s != sphere])
        hard_neg = random.choice(rus_by_sphere[neg_sphere])
        triplets.append({
            "anchor":          anchor['content'][:2000],
            "positive":        positive['content'][:2000],
            "hard_negative":   hard_neg['content'][:2000],
            "anchor_sphere":   sphere,
            "negative_sphere": neg_sphere,
            "pair_type":       "same_sphere_rus",
            "anchor_url":      anchor['url'],
            "positive_url":    positive['url'],
        })

# ── Type 4: same-sphere KAZ ───────────────────────────────────────────────
for sphere in SPHERES:
    pool = kaz_by_sphere[sphere]
    pairs = [(pool[i], pool[j]) for i in range(len(pool)) for j in range(i+1, min(i+4, len(pool)))]
    for anchor, positive in random.sample(pairs, min(250, len(pairs))):
        neg_sphere = random.choice([s for s in SPHERES if s != sphere])
        hard_neg = random.choice(kaz_by_sphere[neg_sphere])
        triplets.append({
            "anchor":          anchor['content'][:2000],
            "positive":        positive['content'][:2000],
            "hard_negative":   hard_neg['content'][:2000],
            "anchor_sphere":   sphere,
            "negative_sphere": neg_sphere,
            "pair_type":       "same_sphere_kaz",
            "anchor_url":      anchor['url'],
            "positive_url":    positive['url'],
        })

random.shuffle(triplets)
print(f"Total triplets generated: {len(triplets)}")
print("By type:", dict(Counter(t['pair_type'] for t in triplets)))

with open("triplets_raw.json", "w", encoding="utf-8") as f:
    json.dump(triplets, f, ensure_ascii=False, indent=2)
print("Saved → triplets_raw.json")


In [ ]:
# Stage 2.5: Content Cleaner

import json, re
from collections import Counter

with open("triplets_raw.json", encoding="utf-8") as f:
    triplets = json.load(f)

print(f"Before cleaning: {len(triplets)} triplets")

# Pure-amendment docs are almost entirely "replace X with Y" or quoted substitutions.
# Detect them by checking if meaningful prose sentences exist.
SENTENCE_RE = re.compile(r'[А-ЯӘІҢҒҮҰҚӨҺа-яәіңғүұқөһA-Za-z]{15,}.*?[.!?]')

def has_prose(text: str) -> bool:
    """True if text contains at least 3 proper sentences with real words."""
    return len(SENTENCE_RE.findall(text)) >= 3

def is_clean(text: str) -> bool:
    if not text:
        return False
    words = text.split()
    if len(words) < 60:       # too short
        return False
    if not has_prose(text):   # pure amendment / citation list
        return False
    return True

def triplet_is_clean(t):
    return is_clean(t["anchor"]) and is_clean(t["positive"]) and is_clean(t["hard_negative"])

# Dedup by anchor+positive URL pair
seen_pairs = set()
clean_triplets = []
for t in triplets:
    if not triplet_is_clean(t):
        continue
    pair_key = (t.get("anchor_url", ""), t.get("positive_url", ""))
    if pair_key in seen_pairs:
        continue
    seen_pairs.add(pair_key)
    clean_triplets.append(t)

print(f"After cleaning:  {len(clean_triplets)} triplets")
print(f"Removed:         {len(triplets) - len(clean_triplets)} ({100*(1 - len(clean_triplets)/len(triplets)):.1f}%)")
print("By type:", dict(Counter(t['pair_type'] for t in clean_triplets)))

with open("triplets_raw.json", "w", encoding="utf-8") as f:
    json.dump(clean_triplets, f, ensure_ascii=False, indent=2)
print("Overwritten → triplets_raw.json")


In [ ]:
# Stage 3: Kimi K2.5 Reasoner via Groq (async, checkpointed)

import json, asyncio, os, re
from groq import AsyncGroq
from dotenv import load_dotenv
load_dotenv()

client = AsyncGroq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "moonshotai/kimi-k2-instruct"

SYSTEM_PROMPT = """You are a legal document analysis expert specializing in Kazakhstani law.
Analyze the semantic relationship between legal documents and output strict JSON only."""

async def label_triplet(triplet, sem):
    prompt = f"""You are given three Kazakhstani legal document excerpts.

ANCHOR (sphere: {triplet['anchor_sphere']}):
{triplet['anchor'][:1200]}

POSITIVE CANDIDATE (sphere: {triplet['anchor_sphere']}):
{triplet['positive'][:1200]}

HARD NEGATIVE (sphere: {triplet['negative_sphere']}):
{triplet['hard_negative'][:800]}

Output strict JSON only (no markdown, no explanation):
{{
  "relationship": "amends|supplements|parallel|contradicts|unrelated",
  "anchor_positive_similarity": 0.0,
  "hard_negative_distinct": true,
  "deprecated": false,
  "confidence": 0.0,
  "reasoning": "one sentence"
}}"""

    async with sem:
        resp = await client.chat.completions.create(
            model=MODEL,
            max_tokens=300,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )
    text = resp.choices[0].message.content.strip()
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group()) if m else None


async def run_labeling(triplets, concurrency=8, checkpoint_every=200,
                       checkpoint_file="triplets_labeled.json"):
    sem = asyncio.Semaphore(concurrency)
    labeled, filtered_out, errors = [], 0, 0

    start = 0
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, encoding="utf-8") as f:
            labeled = json.load(f)
        start = len(labeled)
        print(f"Resuming from checkpoint: {len(labeled)} already labeled")

    TO_LABEL = triplets[start:]

    async def process(t):
        nonlocal filtered_out, errors
        try:
            result = await label_triplet(t, sem)
            if result is None:
                errors += 1
                return
            if (result.get("deprecated")
                    or result.get("confidence", 0) < 0.65
                    or result.get("relationship") == "unrelated"
                    or not result.get("hard_negative_distinct", True)):
                filtered_out += 1
                return
            t["label"] = result
            labeled.append(t)
        except Exception as e:
            errors += 1
            print(f"Error: {e}")

    for batch_start in range(0, len(TO_LABEL), checkpoint_every):
        batch = TO_LABEL[batch_start:batch_start + checkpoint_every]
        await asyncio.gather(*[process(t) for t in batch])
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            json.dump(labeled, f, ensure_ascii=False, indent=2)
        print(f"  [{batch_start + len(batch)}/{len(TO_LABEL)}] kept={len(labeled)} filtered={filtered_out} errors={errors}")

    return labeled


with open("triplets_raw.json", encoding="utf-8") as f:
    triplets = json.load(f)

print(f"Total triplets to label: {min(3000, len(triplets))}")
labeled = await run_labeling(triplets[:3000])

print(f"\nFinal: {len(labeled)} triplets kept")
with open("triplets_labeled.json", "w", encoding="utf-8") as f:
    json.dump(labeled, f, ensure_ascii=False, indent=2)
print("Saved → triplets_labeled.json")


In [ ]:
!pip3 install -q sentence-transformers peft transformers accelerate torch huggingface_hub datasets

In [ ]:
# Download triplets_labeled.json from HuggingFace
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="selffounder/zanembed_v01",
    filename="triplets_labeled.json",
    repo_type="dataset",
)
print("Downloaded to:", path)

import shutil
shutil.copy(path, "triplets_labeled.json")
print("Copied → triplets_labeled.json")


In [ ]:
# Stage 4-5: Triplet Loss + LoRA Training (manual XLA loop)

import json, torch
import torch.nn.functional as F
import torch_xla
import torch_xla.core.xla_model as xm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType

# ── Config ────────────────────────────────────────────────────────────────
BASE_MODEL   = "zeroentropy/zembed-1"
OUTPUT_DIR   = "zanembed-v01-lora"
BATCH_SIZE   = 1
EPOCHS       = 3
WARMUP_STEPS = 100
MAX_SEQ_LEN  = 256
LR           = 2e-4
MARGIN       = 0.3

device = torch_xla.device()
print("Device:", device)

# ── Load triplets ─────────────────────────────────────────────────────────
with open("triplets_labeled.json", encoding="utf-8") as f:
    triplets = json.load(f)

split      = int(len(triplets) * 0.9)
train_data = triplets[:split]
eval_data  = triplets[split:]
print(f"Train: {len(train_data)}  Eval: {len(eval_data)}")

class TripletDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        t = self.data[i]
        return t["anchor"], t["positive"], t["hard_negative"]

# ── Load model + LoRA ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
backbone  = AutoModel.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)
backbone = get_peft_model(backbone, lora_config)
backbone.enable_input_require_grads()
# gradient_checkpointing_enable() removed — references torch.xla which no longer exists in torch_xla 2.x
backbone.print_trainable_parameters()
backbone = backbone.to(device)

# ── Helpers ───────────────────────────────────────────────────────────────
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def encode(texts):
    enc = tokenizer(texts, padding=True, truncation=True,
                    max_length=MAX_SEQ_LEN, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    out = backbone(**enc)
    return mean_pool(out.last_hidden_state, enc["attention_mask"])

def triplet_loss(a, p, n):
    a = F.normalize(a, dim=-1)
    p = F.normalize(p, dim=-1)
    n = F.normalize(n, dim=-1)
    return F.relu((a * n).sum(-1) - (a * p).sum(-1) + MARGIN).mean()

# ── Training loop ─────────────────────────────────────────────────────────
train_loader = DataLoader(TripletDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
optimizer    = AdamW(backbone.parameters(), lr=LR)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

for epoch in range(EPOCHS):
    backbone.train()
    total_loss, count = 0.0, 0
    for step, (anchors, positives, negatives) in enumerate(train_loader):
        optimizer.zero_grad()
        loss = triplet_loss(
            encode(list(anchors)),
            encode(list(positives)),
            encode(list(negatives)),
        )
        loss.backward()
        xm.optimizer_step(optimizer, barrier=True)
        scheduler.step()
        total_loss += loss.item()
        count += 1
        if (step + 1) % 50 == 0:
            print(f"Epoch {epoch+1} | step {step+1}/{len(train_loader)} | loss {total_loss/count:.4f}")
    print(f"Epoch {epoch+1} done — avg loss: {total_loss/count:.4f}")

# ── Save ──────────────────────────────────────────────────────────────────
backbone.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}/")

# ── Sanity check ──────────────────────────────────────────────────────────
backbone.eval()
with torch.no_grad():
    t = triplets[0]
    a = F.normalize(encode([t["anchor"]]),        dim=-1)
    p = F.normalize(encode([t["positive"]]),      dim=-1)
    n = F.normalize(encode([t["hard_negative"]]), dim=-1)
    xm.mark_step()
    print("anchor↔positive sim:", (a * p).sum().item())
    print("anchor↔hard_neg sim:", (a * n).sum().item())


In [ ]:
# Archive the trained LoRA adapter (OUTPUT_DIR from training cell)
import shutil
from pathlib import Path

zip_path = shutil.make_archive("zanembed-v01-lora", "zip", OUTPUT_DIR)
print("Archive:", Path(zip_path).resolve())
# Optional (Google Colab): from google.colab import files; files.download(zip_path)
